# 2048 Expectimax — Heuristic × Depth Experiments

Tests 4 heuristics across depths 1–3 (12 conditions, 25 games each = 300 games total).

**Heuristics:**
- `snake`   — positional snake-pattern weights only
- `empty`   — empty cell count only
- `smooth`  — smoothness (adjacent tile similarity) only
- `combined` — all three together (snake + empty + smooth)

## Run experiments  *(4 heuristics × 3 depths × 25 games = 300 games)*

Progress is printed after every game so you can monitor.

In [1]:
import random
import time

from solve_2048.board import Board
from solve_2048.heuristics import HEURISTICS
from solve_2048.expectimax import MaxNode, expectimax

def play_game(depth, heuristic, seed=None):
    if seed is not None:
        random.seed(seed)

    board = Board()
    board.add_random_tile()
    board.add_random_tile()

    score, moves = 0, 0
    while not board.is_game_over():
        root = MaxNode(board, last_move_score=0)
        _, best_child = expectimax(root, depth, heuristic)
        score += best_child.last_move_score
        board = best_child.board
        board.add_random_tile()
        moves += 1
    max_tile = max(board.grid[r][c] for r in range(4) for c in range(4))
    return {'score': score, 'max_tile': max_tile, 'moves': moves}


N_GAMES = 25
DEPTHS  = [1, 2, 3]
SEEDS   = [i * 37 + 17 for i in range(N_GAMES)]  # 25 reproducible seeds

results = {}
total_conditions = len(HEURISTICS) * len(DEPTHS)
done = 0

for hname, hfn in HEURISTICS.items():
    for depth in DEPTHS:
        key = (hname, depth)
        results[key] = []
        t0 = time.time()
        for i, seed in enumerate(SEEDS):
            game = play_game(depth, hfn, seed=seed)
            results[key].append(game)
            print(f'  [{hname:>8} | depth={depth}] game {i+1}/{N_GAMES}  '
                  f'score={game["score"]:>7,}  max_tile={game["max_tile"]:>5}')
        elapsed = time.time() - t0
        done += 1
        print(f'  --> condition {done}/{total_conditions} done in {elapsed:.1f}s\n')

print('All experiments complete!')

  [   snake | depth=1] game 1/25  score=    756  max_tile=   64
  [   snake | depth=1] game 2/25  score=  1,856  max_tile=  128
  [   snake | depth=1] game 3/25  score=  3,688  max_tile=  256
  [   snake | depth=1] game 4/25  score=  2,080  max_tile=  128
  [   snake | depth=1] game 5/25  score=  4,176  max_tile=  256
  [   snake | depth=1] game 6/25  score=  1,504  max_tile=  128
  [   snake | depth=1] game 7/25  score=  1,800  max_tile=  128
  [   snake | depth=1] game 8/25  score=  3,032  max_tile=  256
  [   snake | depth=1] game 9/25  score=  1,736  max_tile=  128
  [   snake | depth=1] game 10/25  score=  2,004  max_tile=  128
  [   snake | depth=1] game 11/25  score=  1,496  max_tile=  128
  [   snake | depth=1] game 12/25  score=  2,504  max_tile=  256
  [   snake | depth=1] game 13/25  score=  3,852  max_tile=  256
  [   snake | depth=1] game 14/25  score=  5,236  max_tile=  512
  [   snake | depth=1] game 15/25  score=  2,432  max_tile=  128
  [   snake | depth=1] game 16/25 

## Summary table

In [2]:
from statistics import mean, stdev

def summarise(game_list):
    scores = [g['score'] for g in game_list]
    tiles  = [g['max_tile'] for g in game_list]
    return {
        'avg_score':  round(mean(scores)),
        'max_score':  max(scores),
        'min_score':  min(scores),
        'std_score':  round(stdev(scores)) if len(scores) > 1 else 0,
        'avg_tile':   round(mean(tiles)),
        'max_tile':   max(tiles),
        'rate_2048':  f"{sum(1 for t in tiles if t >= 2048)}/{len(tiles)}",
        'rate_1024':  f"{sum(1 for t in tiles if t >= 1024)}/{len(tiles)}",
    }

# Print a readable table
header = f"{'Heuristic':>10} | {'Depth':>5} | {'Avg':>7} | {'Max':>7} | {'Min':>7} | {'Std':>6} | {'MaxTile':>7} | {'>=2048':>6} | {'>=1024':>6}"
print(header)
print('-' * len(header))

summary = {}
for hname in HEURISTICS:
    for depth in DEPTHS:
        key = (hname, depth)
        s = summarise(results[key])
        summary[key] = s
        print(f"{hname:>10} | {depth:>5} | {s['avg_score']:>7,} | {s['max_score']:>7,} | "
              f"{s['min_score']:>7,} | {s['std_score']:>6,} | {s['max_tile']:>7} | "
              f"{s['rate_2048']:>6} | {s['rate_1024']:>6}")

 Heuristic | Depth |     Avg |     Max |     Min |    Std | MaxTile | >=2048 | >=1024
-------------------------------------------------------------------------------------
     snake |     1 |   3,299 |  10,440 |     756 |  1,948 |    1024 |   0/25 |   1/25
     snake |     2 |   3,208 |   7,140 |     740 |  1,562 |     512 |   0/25 |   0/25
     snake |     3 |  35,599 | 111,280 |   3,156 | 22,195 |    8192 |  20/25 |  22/25
     empty |     1 |   2,990 |   6,724 |     788 |  1,356 |     512 |   0/25 |   0/25
     empty |     2 |   2,990 |   6,724 |     788 |  1,356 |     512 |   0/25 |   0/25
     empty |     3 |   9,075 |  15,568 |   3,352 |  3,792 |    1024 |   0/25 |  10/25
    smooth |     1 |   1,974 |   4,724 |     552 |    987 |     512 |   0/25 |   0/25
    smooth |     2 |   2,579 |   7,012 |     660 |  1,502 |     512 |   0/25 |   0/25
    smooth |     3 |   6,633 |  15,980 |   1,740 |  3,909 |    1024 |   0/25 |   5/25
  combined |     1 |   4,570 |   9,084 |   1,992 |  2,

## Copy results for the paper

Run this cell after experiments complete to get a clean paste-ready block of all results.

In [3]:
print('RESULTS FOR PAPER')
print('=' * 60)
for hname in HEURISTICS:
    print(f'\nHeuristic: {hname}')
    for depth in DEPTHS:
        s = summary[(hname, depth)]
        print(f'  Depth {depth}: avg={s["avg_score"]:,}  max={s["max_score"]:,}  '
              f'min={s["min_score"]:,}  std={s["std_score"]:,}  '
              f'max_tile={s["max_tile"]}  2048_rate={s["rate_2048"]}  1024_rate={s["rate_1024"]}')

RESULTS FOR PAPER

Heuristic: snake
  Depth 1: avg=3,299  max=10,440  min=756  std=1,948  max_tile=1024  2048_rate=0/25  1024_rate=1/25
  Depth 2: avg=3,208  max=7,140  min=740  std=1,562  max_tile=512  2048_rate=0/25  1024_rate=0/25
  Depth 3: avg=35,599  max=111,280  min=3,156  std=22,195  max_tile=8192  2048_rate=20/25  1024_rate=22/25

Heuristic: empty
  Depth 1: avg=2,990  max=6,724  min=788  std=1,356  max_tile=512  2048_rate=0/25  1024_rate=0/25
  Depth 2: avg=2,990  max=6,724  min=788  std=1,356  max_tile=512  2048_rate=0/25  1024_rate=0/25
  Depth 3: avg=9,075  max=15,568  min=3,352  std=3,792  max_tile=1024  2048_rate=0/25  1024_rate=10/25

Heuristic: smooth
  Depth 1: avg=1,974  max=4,724  min=552  std=987  max_tile=512  2048_rate=0/25  1024_rate=0/25
  Depth 2: avg=2,579  max=7,012  min=660  std=1,502  max_tile=512  2048_rate=0/25  1024_rate=0/25
  Depth 3: avg=6,633  max=15,980  min=1,740  std=3,909  max_tile=1024  2048_rate=0/25  1024_rate=5/25

Heuristic: combined
  Dept